# GReaT Synthetic v1 Template

A Colab-ready template for generating a synthetic v1 dataset with `be_great` on any tabular CSV dataset. Change the dataset configuration cell when reusing this notebook for a new dataset.

## Configuration

Edit the following cell for a new dataset. The most common changes are `DATASET_NAME` and `DATASET_CSV`.

In [ ]:
# === Edit these lines for a new dataset ===
from pathlib import Path
DATASET_NAME = "california"
DATASET_DISPLAY_NAME = "California Housing"
DATASET_KIND = "california_housing"
DATASET_CSV = "/content/california.csv"
LLM_NAME = "distilgpt2"
EPOCHS = 5
BATCH_SIZE = 16
MAX_ROWS = 5000
N_SAMPLES = 5000
FLOAT_PRECISION = 5
SEED = 42
TEST_SIZE = 0.2
OUTPUT_ROOT = Path("/content") / DATASET_NAME / "outputs" / "synthetic_v1"
LOG_DIR = OUTPUT_ROOT / "log"

## 1. Install Colab Dependencies

Install `be_great` and the supporting packages used by this template.

In [ ]:
!pip -q install git+https://github.com/RuthuHK/BE_GREAT_IMPROVED.git
!pip -q install pandas scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 2.1 MB/s eta 0:00:00


## 2. Import Libraries

Import the libraries needed for dataset loading, training, generation, and file handling.

In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from be_great import GReaT

## 3. Load Dataset

Load any tabular CSV dataset, keep a configurable training subset, and prepare the output paths.

In [ ]:
if DATASET_KIND == "california_housing":
    california = fetch_california_housing()
    df = pd.DataFrame(california.data, columns=california.feature_names)
    df["MedHouseVal"] = california.target
elif DATASET_KIND == "csv":
    df = pd.read_csv(DATASET_CSV)
else:
    raise ValueError(f"Unsupported DATASET_KIND: {DATASET_KIND}")

original_rows = len(df)
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
train_rows = len(train_df)
sample_count = N_SAMPLES

output_dir = OUTPUT_ROOT
log_dir = LOG_DIR
output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
experiment_dir = log_dir

print(f"Loaded dataset: {DATASET_DISPLAY_NAME}")
print(f"Original rows: {original_rows}")
print(f"Training rows: {train_rows}")
print(f"Test rows: {len(test_df)}")
print(f"Synthetic rows to generate: {sample_count}")
print(f"Outputs will be saved in: {output_dir}")

print(df.head())
print(df.shape)

Loaded dataset: California Housing
Original rows: 20640
Training rows: 16512
Test rows: 4128
Synthetic rows to generate: 5000
Outputs will be saved in: /content/california/outputs/synthetic_v1
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  
(20640, 9)


## 4. Initialize GReaT

Create the `be_great` model using `distilgpt2`.

In [ ]:
great = GReaT(
    LLM_NAME,
    experiment_dir=str(experiment_dir),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    float_precision=FLOAT_PRECISION,
)
print(great)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GReaT(llm='distilgpt2', epochs=5, batch_size=16, unfitted, device=unresolved)


## 5. Train GReaT

Fit the model on the selected training subset.

In [ ]:
great.fit(train_df)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'loss': '1.632', 'grad_norm': '2.666', 'learning_rate': '4.516e-05', 'epoch': '0.4845'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.365', 'grad_norm': '2.504', 'learning_rate': '4.032e-05', 'epoch': '0.969'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.291', 'grad_norm': '1.996', 'learning_rate': '1.56e-06', 'epoch': '4.845'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1258', 'train_samples_per_second': '65.6', 'train_steps_per_second': '4.1', 'train_loss': '1.344', 'epoch': '5'}


## 6. Generate Synthetic v1

Sample synthetic rows using `be_great` and ensure the output is at least as large as the original dataset.

In [ ]:
try:
    synthetic_df = great.sample(
        n_samples=sample_count,
        guided_sampling=True,
        device="auto",
    )
except Exception as exc:
    print(f"guided_sampling failed: {exc}")
    print("Falling back to legacy sampling...")
    synthetic_df = great.sample(
        n_samples=sample_count,
        guided_sampling=False,
        device="auto",
    )

print(synthetic_df.head())
print(f"Generated rows: {len(synthetic_df)}")

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  2.1805        36   5.29452    1.09051        1852   1.83762     34.10   
1  4.9355        21   5.38963    0.98466        6963   3.76692     32.70   
2  2.4155        13   4.90345    1.01094         867   7.81362     32.52   
3  2.6353        34   4.80484    1.11892         979   2.65860     34.13   
4  2.8333        16   3.81921    1.03897         941   2.82441     34.16   

   Longitude  MedHouseVal  
0    -118.35        2.922  
1    -117.91        2.852  
2    -117.21        0.596  
3    -118.18        1.250  
4    -118.09        2.531  
Generated rows: 5000


## 7. Post-process and Save Results

Clean the generated dataset, then save the CSV and metadata into the dataset-specific output folder.

In [ ]:
synthetic_df = synthetic_df.drop_duplicates().reset_index(drop=True)

output_csv = output_dir / "cal_synthetic_v1.csv"
output_metadata = log_dir / "cal_synthetic_v1_metadata.json"

synthetic_df.to_csv(output_csv, index=False)
metadata = {
    "dataset": DATASET_DISPLAY_NAME,
    "model": LLM_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "training_rows": int(len(train_df)),
    "synthetic_rows": int(len(synthetic_df)),
    "seed": SEED,
    "original_rows": int(original_rows),
    "test_rows": int(len(test_df)),
    "generated_before_dedup": sample_count,
}
output_metadata.write_text(json.dumps(metadata, indent=2))

print(f"Saved CSV to: {output_csv}")
print(f"Saved metadata to: {output_metadata}")
print(metadata)

Saved CSV to: /content/california/outputs/synthetic_v1/cal_synthetic_v1.csv
Saved metadata to: /content/california/outputs/synthetic_v1/log/cal_synthetic_v1_metadata.json
{'dataset': 'California Housing', 'model': 'distilgpt2', 'epochs': 5, 'batch_size': 16, 'training_rows': 16512, 'synthetic_rows': 5000, 'seed': 42, 'original_rows': 20640, 'test_rows': 4128, 'generated_before_dedup': 5000}
